In [1]:
import torch
import torch.nn as nn
import snntorch as snn
import numpy as np
from snntorch import import_from_nir
import nir

In [2]:
graph = nir.read("nir_examples/cnn_sinabs.nir")
graph.nodes.keys()

dict_keys(['0', '1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9', 'input', 'output'])

In [3]:
net = import_from_nir(graph)

replace rnn subgraph with nirgraph


In [4]:
net

GraphExecutor(
  (0): Conv2d(2, 16, kernel_size=(5, 5), stride=(2, 2), padding=(1, 1))
  (1): Leaky()
  (10): Leaky()
  (11): Linear(in_features=256, out_features=10, bias=True)
  (12): Leaky()
  (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): Leaky()
  (4): AvgPool2d(kernel_size=(2, 2), stride=(2, 2), padding=(0, 0))
  (5): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): Leaky()
  (7): AvgPool2d(kernel_size=(2, 2), stride=(2, 2), padding=(0, 0))
  (8): Flatten(start_dim=0, end_dim=-1)
  (9): Linear(in_features=128, out_features=256, bias=True)
  (input): Identity()
  (output): Identity()
)

## 1) Spike activity of the hidden layer

In [5]:
inp_data = torch.from_numpy(np.load("nir_examples/cnn_numbers.npy")).float()
inp_data.shape

torch.Size([300, 10, 2, 34, 34])

In [6]:
debug_data = inp_data[:3, :1, :, :, :]

In [7]:
print(debug_data.shape)

torch.Size([3, 1, 2, 34, 34])


In [8]:
for t in range(3):
    for s in range(1):
        for c in range(2):
            for i in range(34):
                for j in range(34):
                    if debug_data[t, s, c, i, j] > 0:
                        print(t, s, c, i, j)

0 0 0 11 18
1 0 0 13 21
2 0 0 24 12


In [15]:
tensor_flat = debug_data.reshape(-1)

with open("debug_data.txt", "w") as f:
    f.write(" ".join(str(int(x)) for x in tensor_flat))

In [10]:
modules = [e.elem for e in net.get_execution_order()]

In [11]:
def format_tensor(mat, num_tabs = 0):

    if len(mat.shape) == 1:

        s = "\t" * num_tabs + "{"

        for i in range(mat.shape[0]):
            
            s += f"{mat[i]:.4f}"

            if i < mat.shape[0] - 1:
                s += ", "

        return s + "}"
    
    s = "\t" * num_tabs + "{"

    for i in range(mat.shape[0]):

        s += "\n" + format_tensor(mat[i], num_tabs + 1)

        if i < mat.shape[0] - 1:
            s += ",\n"

    s += "\n" + "\t" * num_tabs + "}"
    return s

In [12]:
# init all I&F neurons
mem_dict = {}
for idx, module in enumerate(modules):
    if isinstance(module, snn.Leaky):
        mem_dict[idx] = module.init_leaky()

out = []
act = []
with open("debug_snntorch.txt", "w", encoding="utf-8") as f:
    for t in range(debug_data.shape[0]):
        x = debug_data[t]
        spklayer = None

        for idx, module in enumerate(modules):

            if isinstance(module, nn.Flatten):
                x = x.flatten(1, -1)

            elif isinstance(module, snn.Leaky):

                f.write("LEAKY:\n")

                # Input
                f.write(f"Input {x.shape}:")
                f.write(format_tensor(x.detach()))
                f.write("\n\n")  # quebras de linha para separar

                # Mem before
                f.write(f"Mem before {mem_dict[idx].shape}:\n")
                f.write(format_tensor(mem_dict[idx].detach()))
                f.write("\n\n")

                # Passa pelo módulo
                x, mem_dict[idx] = module(x, mem_dict[idx])

                # Output
                f.write(f"Output {x.shape}:")
                f.write(format_tensor(x.detach()))
                f.write("\n\n")

                # Mem after
                f.write(f"Mem after {mem_dict[idx].shape}:\n")
                f.write(format_tensor(mem_dict[idx].detach()))
                f.write("\n\n")

                if spklayer is None:
                    spklayer = x.detach().numpy()

            else:
                x = module(x)

        out.append(x)
        act.append(spklayer)

out = torch.stack(out).detach()

In [45]:
print(out.shape)

torch.Size([3, 1, 10])


In [46]:
print(out)

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]])


In [8]:
act_new = np.array(act)
# np.save("snnTorch_activity.npy", act_new)
act_new.shape

(300, 10, 16, 16, 16)

## 2) Accuracy on the whole dataset

In [15]:
import tonic
import tqdm

/home/user3/miniconda3/envs/hls4ml-tutorial/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
bs = 128
collate = tonic.collation.PadTensors(batch_first=False)
to_frame = tonic.transforms.ToFrame(
    sensor_size=tonic.datasets.NMNIST.sensor_size, time_window=1e3
)
test_ds = tonic.datasets.NMNIST("./nmnist", transform=to_frame, train=False)
test_dl = torch.utils.data.DataLoader(
    test_ds, shuffle=True, batch_size=bs, collate_fn=collate
)

accuracies = []
pbar = tqdm.tqdm(total=len(test_dl), desc="Processing", position=0, leave=True)
for idx, (x, y) in enumerate(test_dl):
    # x = torch.moveaxis(x, 0, -1)

    # reset/init I&F neurons
    mem_dict = {}
    for idx, module in enumerate(modules):
        if isinstance(module, snn.Leaky):
            mem_dict[idx] = module.init_leaky()

    # forward pass through time
    out = []
    for t in range(x.shape[0]):
        xt = x[t]
        for idx, module in enumerate(modules):
            if isinstance(module, snn.Leaky):
                xt, mem_dict[idx] = module(xt, mem_dict[idx])
            elif isinstance(module, nn.Flatten):
                xt = xt.flatten(1, -1)
            else:
                xt = module(xt)
        out.append(xt)
    out = torch.stack(out).detach()

    pred = out.mean(0).argmax(-1)
    accuracy = (pred == y).sum() / x.shape[1]
    accuracies.append(accuracy)
    pbar.set_postfix(accuracy="{:.2f}%".format(sum(accuracies) / len(accuracies) * 100))
    pbar.update(1)
pbar.close()
accuracies = np.array(accuracies)
print(f"accuracy: {accuracies.mean():.2%} +/- {accuracies.std():.2%}")
np.save("snntorch_accuracies.npy", accuracies)
np.save("snntorch_accuracy.npy", accuracies.mean())

169675776it [00:53, 3174129.99it/s]                               


Extracting ./nmnist/NMNIST/test.zip to ./nmnist/NMNIST


Processing: 100%|██████████| 79/79 [02:16<00:00,  1.73s/it, accuracy=97.99%]

accuracy: 97.99% +/- 1.25%
